In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import string


from sklearn.model_selection import train_test_split

In [ ]:
# Read the TSV files into the DataFrame
#Data Source Hasso Plattner Institut 
# NDPL - Non-duplicates
# DPL - Duplicates
# ncvoters - a snap shot of the snapshot: VR_Snapshot_20181106 
df_ncvoters = pd.read_csv('/Users/noimotbakare/Dropbox/Mac/Downloads/ncvoters.tsv', sep='\t')
DPL = pd.read_csv('/Users/noimotbakare/Dropbox/Erdös/Erdös_Project_SP2026/Fragmented_ID/ncvoters_DPL.tsv', sep='\t')
NDPL = pd.read_csv('/Users/noimotbakare/Dropbox/Erdös/Erdös_Project_SP2026/Fragmented_ID/ncvoters_NDPL.tsv', sep='\t')



In [ ]:
# Preprocessing for our fragemented ID analysis 
# Selecting identifyer variables not related to voting
df_ncvoters_frag_ID = df_ncvoters[[
#Stable Identifiers
#Only to be used for labeling /evaluation only 
# / not as a model feature
    'id', 'ncid', 'voter_reg_num', 
# Primary Model features - Strongest features, Strong entropy, Essential for matching
    #many variable such as name prefx and sufx are sparse we can either use none for missing or 0/1
    'first_name', 'midl_name', 'last_name', 'name_sufx_cd',
    #other varaiabes
# Adress Similarity features - Address is the second strongest identity anchor, 
    # Street name especially high discriminative signal # Unit numbers distinguish household
    #many variable such as unit designator are sparse we can either use none for missing or 0/1
    'house_num', 'street_name', 'street_dir', 'street_type_cd', 'unit_designator', 'unit_num', 'zip_code', 'res_city_desc',
#other varaiabes
# Demographic agreement indicators - Moderate/Supporting Features 
    # these will help us reduce false matches # they are agreement indicators, low-weight similarity features
    #age group rather than age because grouped/bin age is more stable
    'age', 'age_group', 'sex', 'race_code', 'race_desc', 'ethnic_code', 'ethnic_desc', 'birth_place',
# #other varaiabes 
 'phone_num','area_cd'   
 ]]
# print("\nSelected variables 'A' and 'C':")
print(df_ncvoters_frag_ID)

In [ ]:
# Adding labels to DPL and NDPL
DPL['label'] = 1 
NDPL['label'] = 0


print(DPL.head(10))
print(NDPL.head(10))

In [ ]:
#Merge Duplicate and Non Duplicate pairs
pairs = pd.concat([DPL, NDPL])

print(pairs.head(20))
print(pairs.tail(20))


In [ ]:
# merging ncvoters on DPL and NDPL
pairs = pairs.merge(
    df_ncvoters_frag_ID,
    left_on="id1",
    right_on="id",
    how="left"
)
print(pairs.info())
print(pairs.head(10))

In [ ]:
#merging id2 
pairs = pairs.merge(
    df_ncvoters_frag_ID,
    left_on="id2",
    right_on="id",
    how="left",
    suffixes=("_1", "_2")
)

print(pairs.info())
print(pairs.head(10))

Train Test Val Split

In [ ]:
import networkx as nx
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split 

# only duplicates
dup_pairs = pairs[pairs["label"] == 1]

G = nx.Graph()

# add edges
G.add_edges_from(zip(dup_pairs["id1"], dup_pairs["id2"]))

# connected components = entity clusters
components = list(nx.connected_components(G))

In [ ]:
# Adding ids back in data 

all_ids = set(df_ncvoters_frag_ID["id"])
ids_in_graph = set(G.nodes())

singletons = all_ids - ids_in_graph

for s in singletons:
    components.append({s})

In [ ]:
# Split

train_groups, temp_groups = train_test_split(
    components,
    test_size=0.3,
    random_state=42
)

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.5,
    random_state=42
)

In [ ]:
#Converting groups to ids 
train_ids = set().union(*train_groups)
val_ids   = set().union(*val_groups)
test_ids  = set().union(*test_groups)

In [ ]:
#bringing dups and non dups together

train_pairs = pairs[pairs["id1"].isin(train_ids) & pairs["id2"].isin(train_ids)]

val_pairs = pairs[pairs["id1"].isin(val_ids) & pairs["id2"].isin(val_ids)]

test_pairs = pairs[pairs["id1"].isin(test_ids) & pairs["id2"].isin(test_ids)]

In [ ]:
print("Train:", len(train_pairs))
print("Val:", len(val_pairs))
print("Test:", len(test_pairs))

In [ ]:
print(df_ncvoters_frag_ID.columns)
print(pairs.columns)

print(train_pairs.columns)
print(val_pairs.columns)
print(test_pairs.columns)

# Augmentation and Hard Negative Creation



 Augmented positives
 +  typos
  + nicknames
  + address abbreviation

 Hard negatives
  + same household
  + same same lastname
  + same firstname

Medium negative 
  + same first name negative  

In [ ]:
#Creating a new variable id_to_clusters to be used in augmented training cell - #4 household negatives 
id_to_cluster = {}
for cluster_id, component in enumerate(components):
    for voter_id in component:
        id_to_cluster[voter_id] = cluster_id

# singletons
next_id = len(components)
for v_id in df_ncvoters_frag_ID["id"]:
    if v_id not in id_to_cluster:
        id_to_cluster[v_id] = next_id
        next_id += 1

print(f"Total IDs mapped: {len(id_to_cluster)}")

In [ ]:
# 1. Utilities (names + street logic)

import random

nickname_dict = {
    "william": ["bill","billy","will"],
    "robert": ["bob","bobby","rob"],
    "james": ["jim","jimmy"],
    "john": ["jack","johnny"],
    "elizabeth": ["liz","beth","lizzy"],
    "margaret": ["maggie","meg","peggy"],
    "katherine": ["kate","kathy"],
    "sarah": ["sara"]
}

street_expand = {
    "st":"street",
    "rd":"road",
    "ave":"avenue",
    "dr":"drive",
    "ln":"lane",
    "blvd":"boulevard"
}

def nickname(name):
    n = str(name).lower()
    if n in nickname_dict:
        return random.choice(nickname_dict[n])
    for k,v in nickname_dict.items():
        if n in v:
            return k
    return name


def typo(name):
    name=list(str(name))
    if len(name)<3:
        return "".join(name)
    i=random.randint(0,len(name)-2)
    name[i],name[i+1]=name[i+1],name[i]
    return "".join(name)


def expand_street(st):
    s=str(st).lower()
    return street_expand.get(s,s)

    

In [ ]:

# 2. Duplicate augmentation (label = 1) - This creates multi-field realistic duplicates.

def augment_duplicates(train_pairs):
    augmented = []
    for _, row in train_pairs.iterrows():
        if row["label"] != 1:
            continue
        r = row.copy()
        side1_changed = False
        side2_changed = False

        if random.random() < 0.4:
            if random.random() < 0.5:
                r["first_name_1"] = typo(r["first_name_1"])
                side1_changed = True
            else:
                r["first_name_2"] = typo(r["first_name_2"])
                side2_changed = True

        if random.random() < 0.25:
            if not side1_changed and random.random() < 0.5:
                r["first_name_1"] = nickname(r["first_name_1"])
                side1_changed = True
            elif not side2_changed:
                r["first_name_2"] = nickname(r["first_name_2"])
                side2_changed = True

        if random.random() < 0.25:
            if not side1_changed:
                r["street_type_cd_1"] = expand_street(r["street_type_cd_1"])
            elif not side2_changed:
                r["street_type_cd_2"] = expand_street(r["street_type_cd_2"])

        augmented.append(r)
    return pd.DataFrame(augmented)

#aug_dups2 = augment_duplicates(train_pairs)
aug_dups2 = augment_duplicates(train_pairs)
print(aug_dups2.shape)
print(aug_dups2["first_name_1"].head())
print(train_pairs[train_pairs["label"]==1]["first_name_1"].head())

In [ ]:
#3 adding steert_ type expansion in both direction 

def street_type_negatives(train_pairs, n=2000):
    pairs = []
    for _, row in train_pairs.sample(len(train_pairs)).iterrows():
        if row["label"] == 1:
            continue
        # skip if both street types are empty
        if row["street_type_cd_1"] == "" and row["street_type_cd_2"] == "":
            continue
        r = row.copy()
        if random.random() < 0.5:
            r["street_type_cd_1"] = expand_street(r["street_type_cd_1"])
        else:
            r["street_type_cd_2"] = expand_street(r["street_type_cd_2"])
        if r["street_type_cd_1"] != row["street_type_cd_1"] or \
           r["street_type_cd_2"] != row["street_type_cd_2"]:
            r["label"] = 0
            pairs.append(r)
        if len(pairs) >= n:
            break
    return pd.DataFrame(pairs)

In [ ]:
# ####We wanted: different people, same address --> confusing negative
#Reality:   same address → almost always same person in this dataset


# household_negatives isn't a useful hard negative generator for this specific dataset.
#  # 4. Hard negatives - We generate confusing but non-duplicate pairs. 
# # Household negatives (same address)
# # since this is being generated, I want to make sure to not accidentally append a known duplicate - cluster saftey
# def household_negatives(train_pairs,cluster_map, n=3000):

#     pairs=[]

#     groups=train_pairs.groupby(
#         ["house_num_1","street_name_1"]
#     )

#     for _,g in groups:

#         if len(g)<2:
#             continue

#         sample=g.sample(min(len(g),2))

#         r=sample.iloc[0].copy()

#         #cluster saftey check 
#         if cluster_map.get(sample.iloc[0]["id1"]) == cluster_map.get(sample.iloc[1]["id1"]):
#             continue

#         r["first_name_2"]=sample.iloc[1]["first_name_1"]
#         r["last_name_2"]=sample.iloc[1]["last_name_1"]

#         r["label"]=0
        

#         pairs.append(r)

#         if len(pairs)>=n:
#             break

#     return pd.DataFrame(pairs)

In [ ]:
#5 Nickname negatives 
#this simply adds nickname negatives by swapping some first names to nick names in the NDPL

def nickname_negatives(train_pairs,n=2000):

    pairs=[]

    for _,row in train_pairs.sample(len(train_pairs)).iterrows():
        if row["label"]==1: 
            continue

        r=row.copy()

        r["first_name_2"]=nickname(r["first_name_2"])

        if r["first_name_2"]!=row["first_name_2"]:

            r["label"]=0
            pairs.append(r)

        if len(pairs)>=n:
            break

    return pd.DataFrame(pairs)

In [ ]:
# # Suffix negatives Jr/Sr) - 0 in the negative data so this is meaningless it just would add noise and not signal 
# There 124 dups with suffixes — they're valid duplicates where one record has a missing suffix 
# #generating nonsensical suffix 
# def suffix_negatives(train_pairs,n=1000):

#     pairs=[]

#     for _,row in train_pairs.iterrows():
#          if row["label"] == 1:  # skip duplicates, work from negatives
#             continue
#         r=row.copy()

#         if pd.notna(row["name_sufx_cd_1"]):

#             r["name_sufx_cd_2"]="jr" if row["name_sufx_cd_1"]!="jr" else "sr"
#             r["label"]=0

#             pairs.append(r)

#         if len(pairs)>=n:
#             break

#     return pd.DataFrame(pairs)



Tested Suffix and it looks like it doesn't make sense to generate negative suffixes. 
* Because only dups would have junior-senior pairs. 
* could have non dups that are junior singleton and senior singleton. 
* the occurences of suffixes are sparse in our data
* we have 124 duplicate pairs with suffix mismatch -> valid raw data 

In [ ]:
# # Checking for confusable suffix cases in negatives
# confusable_negs = train_pairs[
#     (train_pairs["label"] == 0) &
#     (train_pairs["first_name_1"] == train_pairs["first_name_2"]) &
#     (train_pairs["last_name_1"] == train_pairs["last_name_2"]) &
#     (train_pairs["name_sufx_cd_1"] != train_pairs["name_sufx_cd_2"])
# ].shape[0]

# # Check confusable suffix cases in duplicates
# confusable_dups = train_pairs[
#     (train_pairs["label"] == 1) &
#     (train_pairs["first_name_1"] == train_pairs["first_name_2"]) &
#     (train_pairs["last_name_1"] == train_pairs["last_name_2"]) &
#     (train_pairs["name_sufx_cd_1"] != train_pairs["name_sufx_cd_2"])
# ].shape[0]

# print(f"Confusable suffix negatives: {confusable_negs}")
# print(f"Confusable suffix duplicates: {confusable_dups}")

# # Check how common suffixes are overall
# print(f"\nSuffix value counts id1:\n{train_pairs['name_sufx_cd_1'].value_counts()}")
# print(f"\nSuffix value counts id2:\n{train_pairs['name_sufx_cd_2'].value_counts()}")

In [ ]:
# #how many are missing suffixes on one side?
# # Verify - how many are just missing suffix on one side
# missing_one_side = suspicious[
#     (suspicious["name_sufx_cd_1"] == "") | 
#     (suspicious["name_sufx_cd_2"] == "")
# ].shape[0]

# actual_conflict = suspicious[
#     (suspicious["name_sufx_cd_1"] != "") & 
#     (suspicious["name_sufx_cd_2"] != "") &
#     (suspicious["name_sufx_cd_1"] != suspicious["name_sufx_cd_2"])
# ].shape[0]

# print(f"Missing suffix on one side: {missing_one_side}")
# print(f"Actual SR/JR conflicts: {actual_conflict}")

In [ ]:
# augmented training set 
aug_dups = augment_duplicates(train_pairs)

# hard 
street_neg = street_type_negatives(train_pairs)
#household_neg = household_negatives(train_pairs, id_to_cluster)
nickname_neg = nickname_negatives(train_pairs)

train_pairs_aug = pd.concat(
    [train_pairs,
     aug_dups, 
     street_neg,
    #household_neg, 
     nickname_neg],
    ignore_index=True
)

In [ ]:
# # adding augmentation to val and test splits 
# def augment_split(pairs):
#     aug = augment_duplicates(pairs)
#     street = street_type_negatives(pairs)
#     nick = nickname_negatives(pairs)
    
#     return pd.concat(
#         [pairs, aug, street, nick],
#         ignore_index=True
#     )

# # Apply consistently across all splits
# # train_pairs_aug = augment_split(train_pairs)
# val_pairs_aug   = augment_split(val_pairs)
# test_pairs_aug  = augment_split(test_pairs)

Checking that all augmentations and hard negatives changes worked on TRAINING SET 

In [ ]:
# 1. Typo check - first_name should differ from original
typo_changes = aug_dups2[aug_dups2["first_name_1"] != train_pairs.loc[aug_dups2.index, "first_name_1"].values]
print(f"Typo applied to first_name_1: {len(typo_changes)}")

typo_changes2 = aug_dups2[aug_dups2["first_name_2"] != train_pairs.loc[aug_dups2.index, "first_name_2"].values]
print(f"Typo applied to first_name_2: {len(typo_changes2)}")

# 2. Nickname check - first_name should be in nickname dict values
nickname_changes1 = aug_dups2[aug_dups2["first_name_1"].str.lower().isin(
    [n for names in nickname_dict.values() for n in names] + list(nickname_dict.keys())
)]
print(f"Nickname in first_name_1: {len(nickname_changes1)}")

nickname_changes2 = aug_dups2[aug_dups2["first_name_2"].str.lower().isin(
    [n for names in nickname_dict.values() for n in names] + list(nickname_dict.keys())
)]
print(f"Nickname in first_name_2: {len(nickname_changes2)}")

# 3. Street expansion check - should see full words not abbreviations
expanded1 = aug_dups2[aug_dups2["street_type_cd_1"].isin(street_expand.values())]
print(f"Street expanded in street_type_cd_1: {len(expanded1)}")

expanded2 = aug_dups2[aug_dups2["street_type_cd_2"].isin(street_expand.values())]
print(f"Street expanded in street_type_cd_2: {len(expanded2)}")

# 4. Nickname negatives check
nick_negs = train_pairs_aug[
    (train_pairs_aug["label"] == 0) &
    (train_pairs_aug["first_name_2"].str.lower().isin(
        [n for names in nickname_dict.values() for n in names]
    ))
]
print(f"Nickname negatives: {len(nick_negs)}")

# 5. Street type negatives check
street_negs = train_pairs_aug[
    (train_pairs_aug["label"] == 0) &
    (train_pairs_aug["street_type_cd_1"].isin(street_expand.values()) |
     train_pairs_aug["street_type_cd_2"].isin(street_expand.values()))
]
print(f"Street type negatives: {len(street_negs)}")

# # 6. Household negatives check
# household_negs = train_pairs_aug[
#     (train_pairs_aug["label"] == 0) &
#     (train_pairs_aug["house_num_1"] == train_pairs_aug["house_num_2"]) &
#     (train_pairs_aug["street_name_1"] == train_pairs_aug["street_name_2"])
# ]
# print(f"Household negatives: {len(household_negs)}")

# 7. Overall label distribution
counts = train_pairs_aug["label"].value_counts()
pcts = train_pairs_aug["label"].value_counts(normalize=True) * 100
print(pd.DataFrame({"count": counts, "percentage": pcts.round(2)}))

In [ ]:
# viewing augmentation results to make 
cols=[
"first_name_1","first_name_2",
"last_name_1","last_name_2",
"house_num_1","house_num_2",
"street_name_1","street_name_2",
"street_type_cd_1","street_type_cd_2",
"label" #, "age"
]

train_pairs_aug[cols].sample(40)

In [ ]:
#Checking for nonesensical dups
train_pairs[train_pairs.label==1][
["first_name_1","first_name_2","last_name_1","last_name_2"]
].sample(40)

In [ ]:
# Check all columns are present in augmented data
print(aug_dups.columns.tolist())

# Spot check - view all fields for a few augmented records
aug_dups.sample(5).T  # .T transposes to see all fields vertically

# Feature Engineering 

In [101]:
from jellyfish import jaro_winkler_similarity
import jellyfish

def build_features(pairs):
    features = pd.DataFrame()
#First name similarity
    features["first_name_sim"] = pairs.apply(
        lambda r: jaro_winkler_similarity(
            str(r["first_name_1"]).lower(), 
            str(r["first_name_2"]).lower()
        ), axis=1
    )
    
    features["first_name_exact"] = (pairs["first_name_1"] == pairs["first_name_2"]).astype(int)


    #Last name similarity
    features["last_name_sim"] = pairs.apply(
        lambda r: jaro_winkler_similarity(
            str(r["last_name_1"]).lower(), 
            str(r["last_name_2"]).lower()
        ), axis=1
    )

    features["last_name_exact"] = (pairs["last_name_1"] == pairs["last_name_2"]).astype(int)



# demographic features
    features["age_diff"] = (
        pd.to_numeric(pairs["age_1"], errors="coerce") -
        pd.to_numeric(pairs["age_2"], errors="coerce")
    ).abs()

    features["age_exact"] = (features["age_diff"] == 0).astype(int)
    features["age_close"] = (features["age_diff"] <= 1).astype(int)
    features["age_grp_match"] = (pairs["age_group_1"] == pairs["age_group_2"]).astype(int)
    
    features["sex_match"] = (pairs["sex_1"] == pairs["sex_2"]).astype(int)

#phone features
    features["area_cd_match"] = (pairs["area_cd_1"] == pairs["area_cd_2"]).astype(int)
 
    features["phone_num_match"] = (pairs["phone_num_1"] == pairs["phone_num_2"]).astype(int)
    
    #Address features
    features["street_name_sim"] = pairs.apply(
        lambda r: jaro_winkler_similarity(
            str(r["street_name_1"]).lower(),
            str(r["street_name_2"]).lower()
        ), axis=1
    )
    
    features["house_num_match"] = (pairs["house_num_1"] == pairs["house_num_2"]).astype(int)

    features["zip_match"] = (pairs["zip_code_1"] == pairs["zip_code_2"]).astype(int)
    
    #Street_type_match
    features["street_type_match"] = (pairs["street_type_cd_1"] == pairs["street_type_cd_2"]).astype(int)

    #features["city_sim"] = (pairs["res_city_desc_1"] == pairs["res_city_desc_2"]).astype(int)
    
#City similarity 
    features["city_sim"] = pairs.apply(
        lambda r: jaro_winkler_similarity(
            str(r["res_city_desc_1"]).lower(),
            str(r["res_city_desc_2"]).lower()
        ), axis=1
    )
    features["zip_code_sim"] = (pairs["zip_code_1"] == pairs["zip_code_2"]).astype(int)

# hard negative signal 

    features["same_addr_diff_name"] = (
    (pairs["house_num_1"] == pairs["house_num_2"]) &
    (pairs["street_name_1"] == pairs["street_name_2"]) &
    (pairs["last_name_1"] != pairs["last_name_2"])
    ).astype(int)

# phonetic features 
    for col in ["first_name", "last_name"]:

        col1 = pairs[f"{col}_1"].fillna("").str.lower()
        col2 = pairs[f"{col}_2"].fillna("").str.lower()

    # encodings
        features[f"{col}_soundex_1"] = pairs[f"{col}_1"].fillna("").apply(jellyfish.soundex)
        features[f"{col}_soundex_2"] = pairs[f"{col}_2"].fillna("").apply(jellyfish.soundex)

        features[f"{col}_metaphone_1"] = pairs[f"{col}_1"].fillna("").apply(jellyfish.metaphone)
        features[f"{col}_metaphone_2"] = pairs[f"{col}_2"].fillna("").apply(jellyfish.metaphone)

    # matches
        features[f"{col}_soundex_match"] = (
            features[f"{col}_soundex_1"] == features[f"{col}_soundex_2"]
        ).astype(int)

        features[f"{col}_metaphone_match"] = (
            features[f"{col}_metaphone_1"] == features[f"{col}_metaphone_2"]
        ).astype(int)


    return features

Passing Engineered Features through train, validation, and test split.

- Note that the augmentation and hard negatives are passed through only the training set. 

- the aug and hard negatives are not  passed through the evaluation set (val and test) per ML rule. 

In [102]:
#Training set contains augmentation/hard negatives 
X_train = build_features(train_pairs_aug)

#unaltered 
X_val   = build_features(val_pairs)
X_test  = build_features(test_pairs)


In [ ]:
#X_train

,first_name_sim,first_name_exact,last_name_sim,last_name_exact,age_diff,age_exact,age_close,age_grp_match,sex_match,area_cd_match,...,first_name_metaphone_1,first_name_metaphone_2,first_name_soundex_match,first_name_metaphone_match,last_name_soundex_1,last_name_soundex_2,last_name_metaphone_1,last_name_metaphone_2,last_name_soundex_match,last_name_metaphone_match
0,1.000000,1,1.000000,1,0,1,1,1,0,0,...,ERK,ERK,1,1,W516,W516,WNBRJR,WNBRJR,1,1
1,0.804286,0,1.000000,1,0,1,1,1,1,0,...,SMJ,SMJ,1,0,G635,G635,KRTN,KRTN,1,1
2,0.971429,0,1.000000,1,0,1,1,1,1,1,...,SMJ,SMJ,1,0,G635,G635,KRTN,KRTN,1,1
3,0.971429,0,1.000000,1,0,1,1,1,1,1,...,SMJ,SMJ,1,0,G635,G635,KRTN,KRTN,1,1
4,0.840000,0,1.000000,1,0,1,1,1,1,0,...,SMJ,SMJ,1,1,G635,G635,KRTN,KRTN,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68393,1.000000,1,0.414286,0,0,1,1,1,0,0,...,JNT,JNT,1,1,G626,W452,KRKR,WLMSN,0,0
68394,1.000000,1,0.414286,0,0,1,1,1,0,0,...,JNT,JNT,1,1,G626,W452,KRKR,WLMSN,0,0
68395,1.000000,1,0.414286,0,0,1,1,1,0,0,...,JNT,JNT,1,1,G626,W452,KRKR,WLMSN,0,0
68396,1.000000,1,0.472222,0,0,1,1,1,1,1,...,STFN,STFN,1,1,R400,G656,RL,KRNR,0,0


In [103]:
y_train = train_pairs_aug["label"]
y_val   = val_pairs["label"]
y_test  = test_pairs["label"]

# Conclusion 
Baseline is performing really well, because the data is still easy! But within the data we know the model struggles to identify duplicates with different last names. So let's focus our attention on that part of our analysis. 

In [68]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train = build_features(train_pairs_aug)
y_train = train_pairs_aug["label"]

X_val = build_features(val_pairs_aug)
y_val = val_pairs_aug["label"]

# X_test = build_features(test_pairs_aug)
# y_test = test_pairs_aug["label"]

model = LogisticRegression()
model.fit(X_train, y_train)

print("Val results:")
print(classification_report(y_val, model.predict(X_val)))

# print("Test results:")
# print(classification_report(y_test, model.predict(X_test)))

TypeError: unsupported operand type(s) for -: 'str' and 'str'

In [ ]:
# from jellyfish import jaro_winkler_similarity

# def build_features(pairs):
#     features = pd.DataFrame()
    
#     # Name similarity - primary signal (almost never differs in real dups)
#     features["first_name_sim"] = pairs.apply(
#         lambda r: jaro_winkler_similarity(
#             str(r["first_name_1"]).lower(), 
#             str(r["first_name_2"]).lower()
#         ), axis=1
#     )
#     features["last_name_sim"] = pairs.apply(
#         lambda r: jaro_winkler_similarity(
#             str(r["last_name_1"]).lower(), 
#             str(r["last_name_2"]).lower()
#         ), axis=1
#     )
    
#     # Address similarity - noisy signal (91% differ in real dups)
#     features["house_num_match"] = (pairs["house_num_1"] == pairs["house_num_2"]).astype(int)
#     features["street_name_sim"] = pairs.apply(
#         lambda r: jaro_winkler_similarity(
#             str(r["street_name_1"]).lower(),
#             str(r["street_name_2"]).lower()
#         ), axis=1
#     )
#     features["street_type_match"] = (pairs["street_type_cd_1"] == pairs["street_type_cd_2"]).astype(int)
    
#     return features

In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import classification_report

# X_train = build_features(train_pairs_aug)
# y_train = train_pairs_aug["label"]

# X_val = build_features(val_pairs)
# y_val = val_pairs["label"]

# model = LogisticRegression()
# model.fit(X_train, y_train)

# y_pred = model.predict(X_val)
# print(classification_report(y_val, y_pred))

# # Check feature importance
# coefficients = pd.DataFrame({
#     "feature": X_train.columns,
#     "coefficient": model.coef_[0]
# }).sort_values("coefficient", ascending=False)
# print(coefficients)

In [ ]:
# print(val_pairs.shape)
# print(val_pairs["label"].value_counts(normalize=True))

# The New Approach:
 We want to test whether the model detects duplicates with different last names without over matching negatives

without over-matching negatives

Overall question: When do traditional linkage models fail, and can deep learning recover those cases?

Given an evaluation set containing: 

* ALL non-duplicates + duplicates with different last names 
* we can add + confusable negatives

Classification task:
* Our model must decide between: hard duplicates (surname change) vs true non-duplicates

Case when: 

* label == 1 AND last_name differs

Our Eval data set will be: 
* label == 1 AND last_name differs 
* we can add + confusable negative (same last name )



 Quick look based on our evaluation above: When we zoom into the actual hard cases (in this case within the validation set) accuracy and recall are .80

In [56]:
# Hard case label is 1, k
hard_dups = val_pairs[
    (val_pairs["label"] == 1) &
    (val_pairs["last_name_1"] != val_pairs["last_name_2"])
]
print(f"Hard duplicate cases in val: {len(hard_dups)}")

y_pred = model.predict(build_features(hard_dups))
print(classification_report(hard_dups["label"], y_pred))

Hard duplicate cases in val: 229
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.80      0.89       229

    accuracy                           0.80       229
   macro avg       0.50      0.40      0.44       229
weighted avg       1.00      0.80      0.89       229



/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

Therefore, our evaluation metric will evaluate where duplicates (label ==1) with last name mismatch, (last_name_1 != last_name_2). 

In [ ]:
# primary evaluation metric going forward should be on 
hard_val = val_pairs[
    (val_pairs["label"] == 1) &
    (val_pairs["last_name_1"] != val_pairs["last_name_2"])
]

In [ ]:
#Hard positives 
recall = (y_pred == 1).mean()
print(f"Recall on hard duplicates: {recall:.2f}")

In [ ]:
hard_dups = val_pairs[
    (val_pairs.label == 1) &
    (val_pairs.last_name_1 != val_pairs.last_name_2)
]

In [54]:
hard_dups_all = pairs[
    (pairs["label"] == 1) &
    (pairs["last_name_1"] != pairs["last_name_2"])
]
print(f"Total hard duplicates: {len(hard_dups_all)}")
print(f"As % of all duplicates: {len(hard_dups_all)/pairs['label'].sum()*100:.1f}%")

# Check how many are in train specifically
hard_dups_train = train_pairs[
    (train_pairs["label"] == 1) &
    (train_pairs["last_name_1"] != train_pairs["last_name_2"])
]

hard_dups_val = val_pairs[
    (val_pairs["label"] == 1) &
    (val_pairs["last_name_1"] != val_pairs["last_name_2"])
]

hard_dups_test = test_pairs[
    (test_pairs["label"] == 1) &
    (test_pairs["last_name_1"] != test_pairs["last_name_2"])
]
print(f"Hard duplicates in train: {len(hard_dups_train)}")
print(f"As % of all hard duplicates: {len(hard_dups_train)/len(hard_dups_all)*100:.1f}%")
print(f"Hard duplicates in val: {len(hard_dups_val)}")
print(f"As % of all hard duplicates: {len(hard_dups_val)/len(hard_dups_all)*100:.1f}%")
print(f"Hard duplicates in test: {len(hard_dups_test)}")
print(f"As % of all hard duplicates: {len(hard_dups_test)/len(hard_dups_all)*100:.1f}%")


Total hard duplicates: 1415
As % of all duplicates: 14.4%
Hard duplicates in train: 965
As % of all hard duplicates: 68.2%
Hard duplicates in val: 229
As % of all hard duplicates: 16.2%
Hard duplicates in test: 221
As % of all hard duplicates: 15.6%


In [58]:
#If we add the confusable negatives

hard_pos = val_pairs[
    (val_pairs["label"] == 1) &
    (val_pairs["last_name_1"] != val_pairs["last_name_2"])
]

confusable_negs = val_pairs[
    (val_pairs["label"] == 0) &
    (val_pairs["last_name_1"] == val_pairs["last_name_2"])
]

hard_val = pd.concat([hard_pos, confusable_negs])

In [59]:
oversample_factor = 3

hard_dups_oversampled = pd.concat(
    [hard_dups_train] * oversample_factor, 
    ignore_index=True
)

train_pairs_aug = pd.concat(
    [train_pairs_aug, hard_dups_oversampled],
    ignore_index=True
)

print(train_pairs_aug["label"].value_counts(normalize=True) * 100)

label
0    75.807772
1    24.192228
Name: proportion, dtype: float64


Logistic regression on hard duplicate last name cases. 

In [60]:
X_train = build_features(train_pairs_aug)
y_train = train_pairs_aug["label"]

model = LogisticRegression()
model.fit(X_train, y_train)

# Primary metric - hard cases
# Keep all val pairs but flag the hard duplicate cases
hard_val = val_pairs[
    ((val_pairs["label"] == 1) & (val_pairs["last_name_1"] != val_pairs["last_name_2"])) |
    (val_pairs["label"] == 0)
]

y_pred = model.predict(build_features(hard_val))
print(classification_report(hard_val["label"], y_pred))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99      2104
           1       0.84      0.99      0.91       229

    accuracy                           0.98      2333
   macro avg       0.92      0.99      0.95      2333
weighted avg       0.98      0.98      0.98      2333



In [ ]:
#Selecting oversampeling percentage
for factor in [2, 3, 5]:
    hard_oversampled = pd.concat([hard_dups_train] * factor, ignore_index=True)
    train_aug = pd.concat([train_pairs_aug, hard_oversampled], ignore_index=True)
    
    model = LogisticRegression()
    model.fit(build_features(train_aug), train_aug["label"])
    
    y_pred = model.predict(build_features(hard_val))
    
    # correct recall calculation
    true_labels = hard_val["label"].values
    recall = ((y_pred == 1) & (true_labels == 1)).sum() / (true_labels == 1).sum()
    print(f"Factor {factor} - hard recall: {recall:.2%}")

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_true, y_pred)

Quick TEST on Diluted Version -  to see if we should include first name difference dups and last name different dups. We shouldn't. Diluted Version of true hard negatives - we will not use this version because duplicate recall ≈ 0.80  for last name not equal and for first name not equal duplicates. This is the same recall for last name not equal duplicates alone. 

* that means the logistic model struggles specifically with surname-change duplicates, so our research question is will focus on last names. 


What actually differ in real duplicates 

* first_name: 139 differ (2.0%) <- almost never differs
* last_name: 965 differ (14.1%) <- Secondary signal 
* house_num: 6230 differ (91.3%)  <- Primary fragmentation signal 
* street_name: 6203 differ (90.9%) <- Primary fragmentation signal 
* street_type_cd: 5082 differ (74.5%) <- Strong signal

This tells us what we knew already from the EDA 
* duplicates in our dataset are primarily people who moved — same person, different address.
* and name changes (marraige, divorce) is a secondary signal


In [ ]:

# hard_name = train_pairs[
#     (train_pairs["label"] == 1) &
#     ((train_pairs["last_name_1"] != train_pairs["last_name_2"]) |
#      (train_pairs["first_name_1"] != train_pairs["first_name_2"]))
# ]
# print(f"Hard cases with name differences: {len(hard_name)}")
# hard_name_val = val_pairs[
#     (val_pairs["label"] == 1) &
#     ((val_pairs["last_name_1"] != val_pairs["last_name_2"]) |
#      (val_pairs["first_name_1"] != val_pairs["first_name_2"]))
# # 

# hard_name_test = test_pairs[
#     (test_pairs["label"] == 1) &
#     ((test_pairs["last_name_1"] != test_pairs["last_name_2"]) |
#      (test_pairs["first_name_1"] != test_pairs["first_name_2"]))
# ]

# print(f"Hard cases in train: {len(hard_name)}")
# print(f"Hard cases in val: {len(hard_name_val)}")
# print(f"Hard cases in test: {len(hard_name_test)}")


* diluted hard case recall:    80%  <- DL model needs to beat this
* diluted hard case precision: 86%  <- DL model needs to beat this
* diluted overall accuracy:    96%
* diluted hard case support:   1072 in train set, 256 in val, 253 in test

In [ ]:

# hard_val_expanded = val_pairs[
#     ((val_pairs["label"] == 1) &
#      ((val_pairs["last_name_1"] != val_pairs["last_name_2"]) |
#       (val_pairs["first_name_1"] != val_pairs["first_name_2"]))) |
#     (val_pairs["label"] == 0)
# ]

# y_pred = model.predict(build_features(hard_val_expanded))
# print(classification_report(hard_val_expanded["label"], y_pred))
# # What actually differs between real duplicates
# orig_dups = train_pairs[train_pairs["label"] == 1]
# cols = ["first_name", "last_name", "house_num", "street_name", "street_type_cd"]

# for col in cols:
#     diff = (orig_dups[f"{col}_1"] != orig_dups[f"{col}_2"]).sum()
#     pct = diff / len(orig_dups) * 100
#     print(f"{col}: {diff} differ ({pct:.1f}%)")
# # Check how many household pairs get rejected by cluster safety check
# groups = train_pairs.groupby(["house_num_1","street_name_1"])
# rejected = 0
# eligible = 0

# for _, g in groups:
#     if len(g) < 2:
#         continue
#     sample = g.sample(min(len(g), 2))
#     eligible += 1
#     if id_to_cluster.get(sample.iloc[0]["id1"]) == id_to_cluster.get(sample.iloc[1]["id1"]):
#         rejected += 1

# print(f"Eligible groups: {eligible}")
# print(f"Rejected by cluster safety check: {rejected}")
# print(f"Passed: {eligible - rejected}")